In [1]:
import pandas as pd

In [2]:
from openai import OpenAI
import os
from dotenv import load_dotenv

# Load .envrc (or .env) file
load_dotenv('../.envrc')
client = OpenAI()

In [3]:
df = pd.read_csv('../data/feature_store_data.csv')
documents = df.to_dict(orient="records")

In [4]:
documents[0]

{'id': 0,
 'feature_name': 'purchase_count_7d',
 'feature_group': 'customer_behavior',
 'computation_logic': "COUNT(order_id) WHERE order_status='DELIVERED' OVER last 7 days BY customer_id",
 'data_source': 'fct_orders (Silver)',
 'update_frequency': 'Hourly',
 'serving_store': 'DynamoDB, S3',
 'models_using_feature': 'recommendation_model, churn_model',
 'feature_description': 'Number of delivered orders placed by a customer in the last 7 days. This feature is computed from fct_orders (Silver) and is commonly used by recommendation_model, churn_model to capture recent customer, product, or operational behavior.'}

In [5]:
prompt_template = """
You emulate a user of our feature store documentation assistant application.
Formulate 5 questions this user might ask based on a provided feature.
Make the questions specific to this feature.
The record should contain the answer to the questions, and the questions should
be complete and not too short. Use as fewer words as possible from the record.

The record:

feature_name: {feature_name}
feature_group: {feature_group}
computation_logic: {computation_logic}
data_source: {data_source}
update_frequency: {update_frequency}
serving_store: {serving_store}
models_using_feature: {models_using_feature}
feature_description: {feature_description}

Provide the output in parsable JSON without using code blocks:

{{"questions": ["question1", "question2", ..., "question5"]}}
""".strip()

In [6]:
prompt = prompt_template.format(**documents[0])

In [7]:
def llm(prompt):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [8]:
questions = llm(prompt)

In [9]:
import json

In [10]:
json.loads(questions)

{'questions': ['What does the purchase_count_7d feature represent in terms of customer behavior?',
  'How is the purchase_count_7d feature calculated based on order data?',
  'What is the frequency of updates for the purchase_count_7d feature in the feature store?',
  'Which data sources are used to compute the purchase_count_7d feature?',
  'What machine learning models utilize the purchase_count_7d feature in their computations?']}

In [11]:
def generate_questions(doc):
    prompt = prompt_template.format(**doc)

    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": prompt}]
    )

    json_response = response.choices[0].message.content
    return json_response

In [12]:
from tqdm.auto import tqdm

In [13]:
results = {}

In [14]:
for doc in tqdm(documents): 
    doc_id = doc['id']
    if doc_id in results:
        continue

    questions_raw = generate_questions(doc)
    questions = json.loads(questions_raw)
    results[doc_id] = questions['questions']

  0%|          | 0/240 [00:00<?, ?it/s]

In [15]:
results[1]

['What does the purchase_count_14d feature represent for a customer?',
 'How is the purchase_count_14d feature calculated based on order data?',
 'What is the source of the data used for the purchase_count_14d feature?',
 'How often is the purchase_count_14d feature updated in the system?',
 'Which models utilize the purchase_count_14d feature for their operations?']

In [16]:
final_results = []

for doc_id, questions in results.items():
    for q in questions:
        final_results.append((doc_id, q))

In [17]:
final_results[0]

(0,
 'What does the purchase_count_7d feature represent in customer behavior analysis?')

In [18]:
df_results = pd.DataFrame(final_results, columns=['id', 'question'])

In [19]:
df_results.to_csv('../data/ground-truth-retrieval.csv', index=False)

In [20]:
!head ../data/ground-truth-retrieval.csv

id,question
0,What does the purchase_count_7d feature represent in customer behavior analysis?
0,How is the purchase_count_7d feature calculated from the data source?
0,What are the update frequency and serving stores for the purchase_count_7d feature?
0,Which models utilize the purchase_count_7d feature for predictions?
0,What specific data source is used to compute the purchase_count_7d feature?
1,What does the purchase_count_14d feature represent for a customer?
1,How is the purchase_count_14d feature calculated based on order data?
1,What is the source of the data used for the purchase_count_14d feature?
1,How often is the purchase_count_14d feature updated in the system?
